In [2]:
import boto3
import re
from datetime import datetime
from dateutil.relativedelta import relativedelta
from botocore import UNSIGNED
from botocore.client import Config

In [3]:
def list_folders_in_range(bucket_name, input_date):
    
    start_date = input_date - relativedelta(months=2)
    end_date = input_date + relativedelta(months=2)
    print(f"folders from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}\n")
    
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    response = s3.list_objects_v2(Bucket=bucket_name)
    
    folders = set()  

    for obj in response.get('Contents', []):
        key = obj['Key']
        parts = key.split('/') #Splitting the date based on the format in folders name
        if len(parts) > 6:
            folder_path = "/".join(parts[:7]) + "/"
            folder_name = parts[6]

            m = re.search(r'(\d{8})', folder_name)
            if m:
                try:
                    folder_date = datetime.strptime(m.group(1), "%Y%m%d")
                except ValueError:
                    continue
            else:

                try:
                    folder_date = datetime.strptime(parts[4] + parts[5].zfill(2) + "01", "%Y%m%d")
                except ValueError:
                    continue
            
            # If the folder date is within the 2-month before/after range, add it
            if start_date <= folder_date <= end_date:
                folders.add(folder_path)

    return sorted(folders)

if __name__ == "__main__":
    bucket = "sentinel-cogs"  
    date_str = input("Enter a date (YYYY-MM-DD): ")
    input_date = datetime.strptime(date_str, "%Y-%m-%d")
    
    
    folders_in_range = list_folders_in_range(bucket, input_date)
    
    if folders_in_range:
        print("Folders in the specified date range:")
        for folder in folders_in_range:
            print(folder)
    else:
        print("No folders found in the given date range.")


Enter a date (YYYY-MM-DD): 2019-09-21
folders from 2019-07-21 to 2019-11-21

Folders in the specified date range:
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191009_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191009_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191019_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191019_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191029_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191029_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/11/S2B_1CCV_20191108_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/11/S2B_1CCV_20191108_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/11/S2B_1CCV_20191118_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/11/S2B_1CCV_20191118_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190909_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190909_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S